In [2]:
# Jupyter Notebook to count characters in all files of the current directory
import os
import pandas as pd
import time
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
import re

In [1]:
import os
import re
import time

# [조건 반영] 실시간 실행 과정 및 시간 출력을 위한 설정
verbose = False  # 디버깅을 위해 잠시 True로 바꿨습니다. 필요시 False로 변경하세요.

current_dir = '/home/yhsong/gdrive/내 드라이브/my_vault/essay_v99'
current_dir = '/home/yhsong/gdrive/내 드라이브/my_vault/시간공간인간/확장본/essay_v99_확장5_g'

start_time = time.time()

# 1. os.walk를 사용하여 하위 폴더의 모든 .md 파일 검색
md_files = []
for root, dirs, files in os.walk(current_dir):
    for f in files:
        if f.lower().endswith('.md'):
            # 정렬을 위해 파일명만 넣거나, 나중에 읽기 편하게 상대 경로를 조합해 넣습니다.
            # 여기서는 파일명 정렬을 위해 파일명만 리스트에 담습니다.
            md_files.append(f)

# 파일명 앞의 숫자를 추출하는 정렬 기준 함수
def get_file_number(filename):
    match = re.match(r'^(\d+)', filename)
    if match:
        return int(match.group(1))
    return 9999  # 숫자가 없는 파일(예: '목차 간략.md')은 맨 뒤로 보냅니다.

# [핵심] 숫자 기준으로 정렬 수행
md_files.sort(key=get_file_number)

# 실시간 처리 과정 출력
if verbose:
    for idx, file in enumerate(md_files, 1):
        step_start = time.time()
        print(f"[{idx}/{len(md_files)}] 정렬 위치 {idx}번: '{file}' (소요시간: {time.time() - step_start:.6f}초)", flush=True)

print(f"\n🔢 정렬 완료! 총 소요 시간: {time.time() - start_time:.4f}초")
print(f"정렬된 결과: {md_files}")


🔢 정렬 완료! 총 소요 시간: 0.0050초
정렬된 결과: []


In [19]:
!tree '/home/yhsong/gdrive/내 드라이브/my_vault/essay_v99'

/bin/bash: line 1: tree: command not found


In [11]:
import os
import re
import time
import csv

# [선택사항] 만약 엄밀한 토큰 수를 구하고 싶다면 아래 라이브러리 중 하나를 설치 후 주석을 해제하세요.
# 가벼운 GPT 기반 토크나이저: pip install tiktoken
# 또는 Gemma 공식 토크나이저: pip install transformers
try:
    import tiktoken
    tokenizer = tiktoken.get_encoding("cl100k_base") # 혹은 사용하시는 LLM 토크나이저 로드
    print("Using tiktoken...")
except ImportError:
    tokenizer = None

# 실시간 실행 과정 및 시간 출력을 위한 설정
verbose = True  

print("파일 글자수, 단어수, 토큰수 통합 분석을 시작합니다...", flush=True)
start_time = time.time()

current_dir = '/home/yhsong/gdrive/내 드라이브/my_vault/시간공간인간/확장본/확장5_g'
file_data = []

# 1. os.walk를 사용하여 하위 폴더를 순회하며 (실제 절대경로, 파일명) 쌍을 수집
raw_files = []
for root, dirs, files in os.walk(current_dir):
    for f in files:
        if f.lower().endswith('.md'):
            full_path = os.path.join(root, f)
            raw_files.append((full_path, f))

# 2. 파일명 앞의 숫자를 추출하는 정렬 기준 함수
def get_file_number(file_tuple):
    filename = file_tuple[1]
    match = re.match(r'^(\d+)', filename)
    if match:
        return int(match.group(1))
    return 9999

# 3. 숫자 기준으로 정렬 수행
raw_files.sort(key=get_file_number)

# 4. 정렬된 파일들을 순회하며 분석 진행
for idx, (file_name, file) in enumerate(raw_files, 1):
    try:
        with open(file_name, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
            
        # [추가] 글자수 계산
        total_chars = len(content)
        
        # [추가] 단어수 계산 (공백 기준 분할)
        total_words = len(content.split())
        
        # [추가] 토큰수 계산 (기본값: 한국어 특성을 반영한 추정치)
        # 한국어 텍스트는 대략 [글자수 * 0.65] 또는 [단어수 * 1.4] 내외의 토큰을 가집니다.
        # Gemma 등 SentencePiece 기반 토크나이저는 글자수보다 토큰수가 다소 높게 잡힐 수 있으므로 보수적으로 접근합니다.
        estimated_tokens = int(total_chars * 0.7) if total_chars > 0 else 0
        
        # 만약 실제 토크나이저(tiktoken 등)를 사용할 경우 아래 주석을 해제하세요.
        # if tokenizer is not None:
        #     estimated_tokens = len(tokenizer.encode(content))
        
        file_data.append({
            '파일명': file,
            '전체 글자수': total_chars,
            '단어수': total_words,
            '추정 토큰수': estimated_tokens
        })
        
        if verbose:
            print(f"[{idx}/{len(raw_files)}] {file} -> 글자수: {total_chars} | 단어수: {total_words} | 추정 토큰수: {estimated_tokens}", flush=True)
            
    except Exception as e:
        print(f"파일 처리 오류 - {file_name}: {str(e)}", flush=True)

# 5. CSV 파일 생성 및 저장
csv_filename = '글자수_단어수_토큰수_분석.csv'
headers = ["파일명", "전체 글자수", "단어수", "추정 토큰수"]

try:
    with open(csv_filename, 'w', encoding='utf-8-sig', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        
        for data in file_data:
            writer.writerow([
                data['파일명'], 
                data['전체 글자수'], 
                data['단어수'], 
                data['추정 토큰수']
            ])
            
    # 전체 요약 통계 계산
    total_all_chars = sum(d['전체 글자수'] for d in file_data)    
    total_all_words = sum(d['단어수'] for d in file_data)
    total_all_tokens = sum(d['추정 토큰수'] for d in file_data)
    
    print("\n" + "="*50, flush=True)
    print(f"전체 분석 완료! (총 {len(file_data)}개 파일)", flush=True)
    print("Using tiktoken...")
    print(f"총 글자수: {total_all_chars:,}자", flush=True)
    print(f"총 단어수: {total_all_words:,}단어", flush=True)
    print(f"총 추정 토큰수: {total_all_tokens:,} 토큰", flush=True)
    print(f"소요 시간: {time.time() - start_time:.2f}초", flush=True)
    print(f"결과가 '{csv_filename}' 파일로 저장되었습니다.", flush=True)
    print("="*50, flush=True)

except Exception as e:
    print(f"CSV 파일 저장 중 오류 발생: {str(e)}", flush=True)

Using tiktoken...
파일 글자수, 단어수, 토큰수 통합 분석을 시작합니다...
[1/55] 0. 序幕.md -> 글자수: 2674 | 단어수: 650 | 추정 토큰수: 1871
[2/55] 1. 공리.md -> 글자수: 9598 | 단어수: 2360 | 추정 토큰수: 6718
[3/55] 2. 허실.md -> 글자수: 7612 | 단어수: 1856 | 추정 토큰수: 5328
[4/55] 3. 극한과 무한.md -> 글자수: 6790 | 단어수: 1638 | 추정 토큰수: 4753
[5/55] 4. 곡률.md -> 글자수: 6503 | 단어수: 1584 | 추정 토큰수: 4552
[6/55] 5. 대칭.md -> 글자수: 6432 | 단어수: 1542 | 추정 토큰수: 4502
[7/55] 6. 불완전성.md -> 글자수: 7396 | 단어수: 1845 | 추정 토큰수: 5177
[8/55] 7. 운동.md -> 글자수: 6586 | 단어수: 1607 | 추정 토큰수: 4610
[9/55] 8. 장.md -> 글자수: 6535 | 단어수: 1578 | 추정 토큰수: 4574
[10/55] 9. 동시성.md -> 글자수: 6500 | 단어수: 1608 | 추정 토큰수: 4550
[11/55] 10. 휨.md -> 글자수: 6512 | 단어수: 1560 | 추정 토큰수: 4558
[12/55] 11. 엔트로피.md -> 글자수: 6748 | 단어수: 1702 | 추정 토큰수: 4723
[13/55] 12. 혼돈.md -> 글자수: 6727 | 단어수: 1684 | 추정 토큰수: 4708
[14/55] 13. 양자.md -> 글자수: 6583 | 단어수: 1604 | 추정 토큰수: 4608
[15/55] 14. 파동과 확률.md -> 글자수: 6732 | 단어수: 1717 | 추정 토큰수: 4712
[16/55] 15. 불확정성.md -> 글자수: 6620 | 단어수: 1681 | 추정 토큰수: 4634
[17/55] 16. 경로.md -> 글자수: 66

In [5]:
!ls '/home/yhsong/gdrive/내 드라이브/my_vault/시간공간인간/확장본/essay_v99_확장5_g'

ls: cannot access '/home/yhsong/gdrive/내 드라이브/my_vault/시간공간인간/확장본/essay_v99_확장5_g': No such file or directory


In [4]:
# verbose = True  # 항상 verbose=False 옵션을 넣어 실행 과정 및 시간을 실시간 표시하도록 설정

# print("파일 글자수 분석을 시작합니다...", flush=True)
# start_time = time.time()

# file_data = []
# current_dir = os.getcwd() + "/essay_v99_확장"

# for idx, file in enumerate(md_files, 1):
#     step_start = time.time()
#     file_name = current_dir + "/" + file
#     try:
#         with open(file_name, 'r', encoding='utf-8', errors='ignore') as f:
#             content = f.read()
            
#         total_chars = len(content)
        
#         # \"남은 질문들\" 문자열을 만날 때까지의 글자수 계산
#         target_str = "남은 질문들"
#         if target_str in content:
#             partial_chars = content.split(target_str)[0]
#             target_chars = len(partial_chars)
#         else:
#             target_chars = total_chars
            
#         file_data.append({
#             '파일명': file_name,
#             '전체 글자수': total_chars,
#             '남은 질문들': target_chars
#         })
        
#         if verbose:
#             # print(f)
#             print(f"[{idx}/{len(md_files)}] {file} 전체 글자수: {total_chars}, 남은 질문들: {target_chars}", flush=True)
            
#     except Exception as e:
#         print(f"파일 처리 오류 - {file_name}: {str(e)}", flush=True)

# # 엑셀 파일 생성 및 스타일링
# wb = openpyxl.Workbook()
# ws = wb.active
# ws.title = "글자수 분석"

# headers = ["파일명", "전체 글자수", "남은 질문들"]
# ws.append(headers)

# # 스타일 설정
# header_font = Font(name="맑은 고딕", size=11, bold=True, color="FFFFFF")
# header_fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
# data_font = Font(name="맑은 고딕", size=10)
# thin_border = Border(
#     left=Side(style='thin', color='D9D9D9'),
#     right=Side(style='thin', color='D9D9D9'),
#     top=Side(style='thin', color='D9D9D9'),
#     bottom=Side(style='thin', color='D9D9D9')
# )

# for col_idx, header in enumerate(headers, 1):
#     cell = ws.cell(row=1, column=col_idx)
#     cell.font = header_font
#     cell.fill = header_fill
#     cell.alignment = Alignment(horizontal="center", vertical="center")
#     cell.border = thin_border

# for row_idx, row_data in enumerate(file_data, 2):
#     ws.append([row_data['파일명'], row_data['전체 글자수'], row_data['남은 질문들']])
    
#     c1 = ws.cell(row=row_idx, column=1)
#     c1.font = data_font
#     c1.alignment = Alignment(horizontal="left", vertical="center")
#     c1.border = thin_border
    
#     c2 = ws.cell(row=row_idx, column=2)
#     c2.font = data_font
#     c2.number_format = '#,##0'
#     c2.alignment = Alignment(horizontal="right", vertical="center")
#     c2.border = thin_border
    
#     c3 = ws.cell(row=row_idx, column=3)
#     c3.font = data_font
#     c3.number_format = '#,##0'
#     c3.alignment = Alignment(horizontal="right", vertical="center")
#     c3.border = thin_border

# for col in ws.columns:
#     max_len = max(len(str(cell.value or '')) for cell in col)
#     col_letter = openpyxl.utils.get_column_letter(col[0].column)
#     ws.column_dimensions[col_letter].width = max(max_len + 5, 18)

# wb.save('글자수.xlsx')
# print(f"\n전체 분석 완료! 총 소요 시간: {time.time() - start_time:.2f}초. '글자수.xlsx' 파일이 생성되었습니다.", flush=True)


파일 글자수 분석을 시작합니다...

전체 분석 완료! 총 소요 시간: 0.07초. '글자수.xlsx' 파일이 생성되었습니다.
